In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 — Silver: 5-Minute Windowed Aggregation
# MAGIC Reads Bronze as a stream, computes rolling 5-minute averages per device.

# COMMAND ----------

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

# COMMAND ----------

from iot_pipeline.config import BRONZE_TABLE, SILVER_TABLE, CHECKPOINT_BASE
from iot_pipeline.transforms import parse_and_window

# COMMAND ----------

checkpoint_path = CHECKPOINT_BASE + "silver/"

bronze_stream = spark.readStream.table(BRONZE_TABLE)
silver_stream = parse_and_window(bronze_stream)

def upsert_to_silver(micro_batch_df, batch_id):
    spark = micro_batch_df.sparkSession
    if not spark.catalog.tableExists(SILVER_TABLE):
        micro_batch_df.write.format("delta").saveAsTable(SILVER_TABLE)
    else:
        micro_batch_df.createOrReplaceTempView("updates")
        spark.sql(f"""
            MERGE INTO {SILVER_TABLE} AS target
            USING updates AS source
            ON target.window = source.window AND target.device_id = source.device_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

query = (silver_stream.writeStream
    .outputMode("update")
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

# COMMAND ----------

# MAGIC %md ## Validation

# COMMAND ----------

count = spark.table(SILVER_TABLE).count()
assert count > 0, f"{SILVER_TABLE} is empty"
print(f"✅ {SILVER_TABLE}: {count} rows")
print("Expected: 240 (one row per 5-minute window × device). If you see noticeably more, see Part 14's troubleshooting note in the guide.")